# CUAD-QA Unsloth (Llama-3.2-3B)

**Upload this notebook only.** Data comes from Hugging Face, not your Mac.

1. Hugging Face → Settings → Access Tokens → create a token that can **read** `Aby-ss/ma-extraction-3B-research` (add **write** if you want to save the adapter to the Hub).
2. Colab → Runtime → Change runtime type → **T4 GPU**.
3. File → Upload this `.ipynb` (nothing else).
4. Runtime → Run all. Paste `hf_...` when asked. If Hub returns 401, still run the **next** cell — it builds CUAD inside Colab.
5. Wait 30–90 minutes. Last cell downloads `lora_cuad_qa.zip` to your computer.

If you already hit `OutOfMemoryError`: **Runtime → Disconnect and delete runtime**, reconnect as T4, upload this updated notebook, Run all. Do not continue in the crashed session.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then Runtime → Restart session"
print("GPU:", torch.cuda.get_device_name(0))
!pip install -q unsloth datasets trl huggingface_hub tiktoken

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download, login
from pathlib import Path

HF_REPO = "Aby-ss/ma-extraction-3B-research"
token = getpass("Hugging Face token (hf_...): ").strip()
login(token=token, add_to_git_credential=False)

train_path = valid_path = None
try:
    train_path = Path(hf_hub_download(HF_REPO, "splits/cuad_qa_train.jsonl.gz", repo_type="dataset", token=token))
    valid_path = Path(hf_hub_download(HF_REPO, "splits/cuad_qa_valid.jsonl.gz", repo_type="dataset", token=token))
    print("Loaded splits from Hub:", train_path, valid_path)
except Exception as exc:
    print("Hub download failed (401 = not logged in, or token cannot see the private dataset).")
    print("That is OK. Run the next cell to convert CUAD inside this Colab VM, then train.")
    print(exc)
    train_path = valid_path = None

In [ ]:
# Skip this cell if the previous cell already printed Hub paths.
if train_path is None:
    import gzip, json, random, re, zipfile
    from urllib.request import urlretrieve
    import tiktoken

    CUAD_CLAUSE_TYPES = [
        "Document Name", "Parties", "Agreement Date", "Effective Date", "Expiration Date",
        "Renewal Term", "Notice to Terminate Renewal", "Governing Law", "Most Favored Nation",
        "Non-Compete", "Exclusivity", "No-Solicit of Customers", "Competitive Restriction Exception",
        "No-Solicit of Employees", "Non-Disparagement", "Termination for Convenience",
        "Right of First Refusal, Offer or Negotiation (ROFR/ROFO/ROFN)", "Change of Control",
        "Anti-Assignment", "Revenue/Profit Sharing", "Price Restriction", "Minimum Commitment",
        "Volume Restriction", "IP Ownership Assignment", "Joint IP Ownership", "License Grant",
        "Non-Transferable License", "Affiliate IP License-Licensor", "Affiliate IP License-Licensee",
        "Unlimited/All-You-Can-Eat License", "Irrevocable or Perpetual License", "Source Code Escrow",
        "Post-Termination Services", "Audit Rights", "Uncapped Liability", "Cap on Liability",
        "Liquidated Damages", "Warranty Duration", "Insurance", "Covenant Not to Sue",
        "Third Party Beneficiary",
    ]
    ALIASES = {
        "rofrroforofn": "Right of First Refusal, Offer or Negotiation (ROFR/ROFO/ROFN)",
        "noticeperiodtoterminaterenewal": "Notice to Terminate Renewal",
        "noticetoterminaterenewal": "Notice to Terminate Renewal",
        "pricerestrictions": "Price Restriction", "pricerestriction": "Price Restriction",
        "affiliatelicenselicensor": "Affiliate IP License-Licensor",
        "affiliateiplicenselicensor": "Affiliate IP License-Licensor",
        "affiliatelicenselicensee": "Affiliate IP License-Licensee",
        "affiliateiplicenselicensee": "Affiliate IP License-Licensee",
        "unlimitedallyoucaneatlicense": "Unlimited/All-You-Can-Eat License",
        "ipownershipassignment": "IP Ownership Assignment",
        "caponliability": "Cap on Liability",
    }
    SYSTEM = (
        "You are senior M&A counsel reviewing a contract excerpt.\n"
        "Extract every clause that matches the 41 CUAD types listed in the user message.\n"
        "Rules:\n- exact_quote MUST be a verbatim substring of the excerpt.\n"
        "- If a type is absent, omit it.\n- confidence is in [0, 1].\n"
        "- statute_code is an empty string unless a statute is named.\n"
        "Return JSON only: {\"clauses\": [...]}"
    )

    def norm(text):
        return re.sub(r"[^a-z0-9]+", "", text.lower())

    lookup = {norm(n): n for n in CUAD_CLAUSE_TYPES}
    lookup.update(ALIASES)

    zip_path = Path("cuad_data.zip")
    if not zip_path.exists():
        urlretrieve("https://github.com/TheAtticusProject/cuad/raw/main/data.zip", zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        payload = json.loads(zf.read("train_separate_questions.json"))
    docs = []
    for item in payload["data"]:
        context = item["paragraphs"][0]["context"]
        spans = []
        for qa in item["paragraphs"][0]["qas"]:
            m = re.search(r'related to "([^"]+)"', qa["question"])
            raw = m.group(1) if m else qa["question"]
            clause_type = lookup.get(norm(raw))
            if not clause_type:
                raise SystemExit(qa["question"][:80])
            for ans in qa.get("answers") or []:
                q = (ans.get("text") or "").strip()
                if q:
                    spans.append((clause_type, q))
        docs.append({"id": item["title"], "text": context, "spans": spans})

    rng = random.Random(42)
    ids = sorted(d["id"] for d in docs)
    rng.shuffle(ids)
    n_valid = max(1, round(len(ids) * 0.10))
    split_of = {i: ("valid" if n < n_valid else "train") for n, i in enumerate(ids)}
    enc = tiktoken.get_encoding("cl100k_base")
    instruction = (
        "Extract CUAD clause spans from this contract excerpt. "
        "Return JSON {\"clauses\": [{\"clause_type\", \"exact_quote\", \"confidence\", \"statute_code\"}]}. "
        "exact_quote must be copied verbatim. Allowed clause_type values:\n"
        + "\n".join(f"- {n}" for n in CUAD_CLAUSE_TYPES)
    )

    def chunks(text):
        toks = enc.encode(text)
        if len(toks) <= 2048:
            return [text]
        out, start = [], 0
        while start < len(toks):
            out.append(enc.decode(toks[start:start + 2048]))
            if start + 2048 >= len(toks):
                break
            start += 1792
        return out

    rows = {"train": [], "valid": []}
    for doc in docs:
        split = split_of[doc["id"]]
        for idx, piece in enumerate(chunks(doc["text"])):
            seen, clauses = set(), []
            for ctype, quote in doc["spans"]:
                if (ctype, quote) in seen:
                    continue
                if quote not in piece and " ".join(quote.split()) not in " ".join(piece.split()):
                    continue
                seen.add((ctype, quote))
                clauses.append({"clause_type": ctype, "exact_quote": quote, "confidence": 1.0, "statute_code": ""})
            payload = json.dumps({"clauses": clauses}, ensure_ascii=False)
            rows[split].append({
                "id": f"cuad:{doc['id']}::{idx}",
                "messages": [
                    {"role": "system", "content": SYSTEM},
                    {"role": "user", "content": instruction + "\n\nExcerpt:\n" + piece},
                    {"role": "assistant", "content": payload},
                ],
            })

    def write_gz(path, data):
        with gzip.open(path, "wt", encoding="utf-8") as f:
            for row in data:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    train_path, valid_path = Path("cuad_qa_train.jsonl.gz"), Path("cuad_qa_valid.jsonl.gz")
    write_gz(train_path, rows["train"])
    write_gz(valid_path, rows["valid"])
    print(f"converted train={len(rows['train'])} valid={len(rows['valid'])}")
    try:
        api = HfApi()
        api.upload_file(str(train_path), "splits/cuad_qa_train.jsonl.gz", HF_REPO, repo_type="dataset",
                        commit_message="CUAD-QA train split from Colab")
        api.upload_file(str(valid_path), "splits/cuad_qa_valid.jsonl.gz", HF_REPO, repo_type="dataset",
                        commit_message="CUAD-QA valid split from Colab")
        print("Uploaded splits to Hub.")
    except Exception as exc:
        print("Could not upload (token may be read-only). Training will still use local Colab files.", exc)

In [ ]:
import gzip, json, os, torch
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
# Chunks are 2048 tokens plus a long system/instruction. 4096 + mid-train eval OOMs a T4 (16GB).
MAX_SEQ_LENGTH = 2048

def load_jsonl_gz(path):
    rows = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return Dataset.from_list(rows)

train_ds = load_jsonl_gz(train_path)
print(f"train={len(train_ds)} (no eval during training — saves T4 VRAM)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        for messages in examples["messages"]
    ]
    return {"text": texts}

train_ds = train_ds.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=None,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.05,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="no",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        output_dir="outputs_cuad_qa",
        seed=42,
        optim="adamw_8bit",
        fp16=True,
        gradient_checkpointing=True,
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
trainer.train()
model.save_pretrained("lora_cuad_qa")
tokenizer.save_pretrained("lora_cuad_qa")
print("Saved LoRA adapter to ./lora_cuad_qa")

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("lora_cuad_qa", "zip", "lora_cuad_qa")
files.download("lora_cuad_qa.zip")